In [ ]:
from pyscripts.AutoEncoder_Experiment import *
import sys, jax, json
import jax.numpy as jnp
from datetime import datetime

import pyscripts.jax_amber3 as ja
import pyscripts.training_functions as tf

jax.config.update("jax_enable_x64", True)

In [ ]:
json_fn = 'json_inputs/testing.json'
with open(json_fn, 'r') as g:
        json_params = json.load(g)

In [ ]:
#Main uses structural loss immediately
#experiment = AutoEncoder_Experiment(json_fn).main()

In [ ]:
#ACS _RPT
print('Main Invoked with the following params:')
self = AutoEncoder_Experiment(json_fn)

num_init_epochs = self.json_params["training"]["epoch"]["num_init_epochs"]
struct_cutoff = self.json_params["training"]["epoch"]["struct_cutoff"]
scaling_cutoff = self.json_params["training"]["epoch"]["scaling_cutoff"]
final_cutoff = self.json_params["training"]["epoch"]["max_epoch"]

struct_thresh = self.json_params["training"]["thresh"]["structural_go_to_scaling"]
final_thresh = self.json_params["training"]["thresh"]["final_to_end"]

print(f"Epochs - Init: {num_init_epochs}, Struct: {struct_cutoff}, Scale: {scaling_cutoff}, Final {final_cutoff}")
print(f"Threshholds - Struct {struct_thresh}, Final {final_thresh}")

#INIT EPOCHS
print('##############################')
print('#####', f'Init {self.epoch:05d}', '#####')
print('##############################')
end_init_epoch = self.train_nepochs(self.math.atom_rmsd, 'mean', num_init_epochs, nan_check_ind=-4)

# REACH THRESHOLD BY STRUCTURE ALONE
print('##############################')
print('#####', f'Strt {self.epoch:05d}', '#####')
print('##############################')
end_struct_epoch = self.train_to_threshold(self.math.atom_rmsd, 'mean', struct_thresh,
                                           struct_cutoff, potential_coefficient=0, nan_check_ind=-4)

# SCALE IN POTENTIAL ENERGY
print('##############################')
print('#####', f'StSc {self.epoch:05d}', '#####')
print('##############################')
end_scaling_epoch = self.train_scaling_coef(self.math.summation_distance, 'mean', scaling_cutoff)

# REACH A THRESHOLD OF LOSS AGAIN
print('##############################')
print('#####', f'Scld {self.epoch:05d}', '#####')
print('##############################')
end_training_epoch = self.train_to_threshold(self.math.summation_distance, 'mean', final_thresh, final_cutoff)

# DONE REPORT THE LAST EPOCH
print('##############################')
print('#####', f'Done {self.epoch:05d}', '#####')
print('##############################')

In [ ]:
#1 No RMTD, RMSD with new NN (Sigmoids) (Copy ACS Procedure, not NN)
json_fn = '1_RMSD_SigA.json'
with open(json_fn, 'r') as g:
        json_params = json.load(g)

num_init_epochs = json_params["training"]["epoch"]["num_init_epochs"]
struct_cutoff = json_params["training"]["epoch"]["struct_cutoff"]
scaling_cutoff = json_params["training"]["epoch"]["scaling_cutoff"]
final_cutoff = json_params["training"]["epoch"]["max_epoch"]

struct_thresh = json_params["training"]["thresh"]["structural_go_to_scaling"]
final_thresh = json_params["training"]["thresh"]["final_to_end"]

print(f"Epochs - Init: {num_init_epochs}, Struct: {struct_cutoff}, Scale: {scaling_cutoff}, Final {final_cutoff}")
print(f"Threshholds - Struct {struct_thresh}, Final {final_thresh}")

experiment = AutoEncoder_Experiment(json_fn)

print('##############################')
print('#####', f'Init {experiment.epoch:05d}', '#####')
print('##############################')
end_init_epoch = experiment.train_nepochs(tf.rmsd_rng_step, num_init_epochs,
                                          coefficients=[1,0], nan_check_ind=-4)
print('##############################')
print('#####', f'Strt {experiment.epoch:05d}', '#####')
print('##############################')
end_struct_epoch = experiment.train_to_threshold(tf.rmsd_rng_step, struct_thresh, struct_cutoff,
                                                 coefficients=[1,0], nan_check_ind=-4)
print('##############################')
print('#####', f'StSc {experiment.epoch:05d}', '#####')
print('##############################')
end_scaling_epoch = experiment.train_scaling_coef(training_functions.summation_rng_step, scaling_cutoff,
                                                  1, coefficients=[0,0], nan_check_ind=-1)
print('##############################')
print('#####', f'Scld {experiment.epoch:05d}', '#####')
print('##############################')
end_training_epoch = experiment.train_to_threshold(training_functions.summation_rng_step, final_thresh, final_cutoff,
                                                   coefficients=[0, 1], nan_check_ind=-1)
print('##############################')
print('#####', f'Done {experiment.epoch:05d}', '#####')
print('##############################')

In [ ]:
#2 Structural Loss Function With ReLus, invoke the ACS json with main train
#json_fn = '2_Str_ReLu.json'
#with open(json_fn, 'r') as g:
#        json_params = json.load(g)

experiment = AutoEncoder_Experiment(json_fn)

experiment.run_main()

In [ ]:
import jax.numpy as jnp

In [ ]:
def tha_mean(function):


def my_test_func(a, a0, k):
    ener = 0.5*k*(a-a0)**2
    return ener

In [ ]:
data, rest, = jnp.array((1.05, 2.3, 2.96)), jnp.array((1, 2, 3))

In [ ]:
my_test_func(data, rest, 2).mean()

In [ ]:
rng_init = jax.random.PRNGKey(54)
rng, key = jax.random.split(rng_init)
recon, latents = experiment.state.apply_fn({'params':experiment.state.params}, experiment.test_data, rng)

In [ ]:
for i in range(latents.shape[-1]):
    plt.clf()
    plt.title(f'Latent {i}')
    _ = plt.hist(latents[:, i], bins=100)
    plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture

In [ ]:
ics = []
for i in range(1, 20):
    X = np.array(latents)
    MM = GaussianMixture(n_components=i).fit(X)
    ics.append((i, MM.aic(X), MM.bic(X)))
ics = np.array(ics)
plt.clf()
_ = plt.plot(ics[:, 0], ics[:, 1])
_ = plt.plot(ics[:, 0], ics[:, 2])
plt.legend(('Akaike Info Criterion', 'Bayes Info Criterion'))
plt.xlabel('Num Components')
plt.show()

In [ ]:
X = np.array(latents)
MM = GaussianMixture(n_components=12).fit(X) #chosen based on above graph
samples = MM.sample(600)[0]

In [ ]:
for i in range(samples.shape[-1]):
    plt.clf()
    _ = plt.hist(latents[:,i], bins=100, histtype='step', color='g')
    _ = plt.hist(samples[:,i], bins=100, histtype='step', color='b')
    plt.legend(('Reconstructed from Input', 'Sampled from Mixture Model'))
    plt.title(f'Latent {i+1}')
    plt.show()

In [ ]:
import scipy
n_L = latents.shape[-1]
fig, axs = plt.subplots(n_L, n_L, figsize=(15, 10), sharex='col', sharey='row')
for i in range(n_L):
    for j in range(n_L):
        axs[i,j].scatter(latents[:,j], latents[:,i])
        print(i, j, scipy.stats.pearsonr(latents[:,j], latents[:,i]))
fig.savefig('7L.png', dpi=600)

In [ ]:
import jax_amber3 as ja
gas_fun, tors_fun = ja.get_amber_functions('Simulation/ala_deca_peptide.prmtop')

In [ ]:
plt.clf()

recon_energies = gas_fun(recon)

decoded_samples = experiment.model.apply({'params': experiment.state.params}, samples, rng, method=experiment.model.decode)
sampled_energies = gas_fun(decoded_samples)

org_energies = gas_fun(experiment.test_data)

threshold = 1e3
num_excluded_gmm = len(sampled_energies) - len(sampled_energies[sampled_energies<threshold])
num_excluded_nn  = len(recon_energies) - len(recon_energies[recon_energies<threshold])
print(f'Excluding Outliers GMM - {num_excluded_gmm} RECON - {num_excluded_nn}')

_ = plt.hist(org_energies[org_energies < threshold], bins=100, histtype='step', color='r')
_ = plt.hist(recon_energies[recon_energies<threshold], bins=300, histtype='step', color='g')
_ = plt.hist(sampled_energies[sampled_energies<threshold], bins=300, histtype='step', color='b')

plt.legend(('Test Data Potential', 'Reconstructed from Test', 'Sampled from Mixture Model'))
plt.xlabel('Energy (kJ/mol)')
plt.title('Comparison of Energies - GMM and NN')
plt.show()

In [ ]:
plt.clf()
_ = plt.hist(org_energies, bins=100, histtype='step', color='r')
plt.show()
plt.clf()
_ = plt.hist(recon_energies, bins=100, histtype='step', color='g')
plt.show()
plt.clf()
_ = plt.hist(sampled_energies, bins=100, histtype='step', color='b')
plt.show()

In [ ]:
def write_traj(filename, traj_xyz): #(n conf, n_atoms*3) OR (n conf, n_atoms, 3)
        if traj_xyz.shape[-1] != 3:
            traj_xyz = traj_xyz.reshape(traj_xyz.shape[0], -1, 3)
        with md.formats.DCDTrajectoryFile(filename, 'w') as f:
            f.write(traj_xyz*10) #*10 because mdtraj loads data in nm but saves it in angstrom

In [ ]:
my_dict = {'DA_TestData.dcd' : experiment.test_data, 'DA_ReconData.dcd' : recon, 'DA_GMM.dcd' : decoded_samples}
for key, value in zip(my_dict.keys(), my_dict.values()):
    write_traj(key, value)